# ML-09 — Validation and Research Claim Audit

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/jerovernay/FlyRank-Internship/blob/main/work/notebooks/w06_validation_audit.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [1]:
# Shared setup, mirrors w05_model.ipynb's constants/cache/HF connection so this notebook
# reproduces from a clean checkout without re-running w05 first.
import os, warnings
import numpy as np
import pandas as pd
warnings.filterwarnings("ignore", message=".*valid feature names.*")

SEED, VOL_FLOOR, N_FOLDS, K_CLUSTERS = 42, 100, 5, 4
FEATURES = ["log_impressions", "log_clicks", "ctr", "avg_position"]
OUTD = "../outputs"
os.makedirs(OUTD, exist_ok=True)

MARCH_CACHE = f"{OUTD}/w05_march_features.parquet"
BASE = "hf://datasets/FlyRank/internship-warehouse"


def _hf_con():
    """DuckDB + HF secret. Token from a local gitignored .env or the environment - never a cell."""
    import duckdb
    if "HF_TOKEN" not in os.environ and os.path.exists("../../.env"):
        for line in open("../../.env"):
            if line.startswith("HF_TOKEN"):
                os.environ["HF_TOKEN"] = line.strip().split("=", 1)[1]
    con = duckdb.connect()
    con.execute(f"CREATE OR REPLACE SECRET hf (TYPE huggingface, TOKEN '{os.environ['HF_TOKEN']}')")
    return con


assert os.path.exists(MARCH_CACHE), "run w05_model.ipynb once first to populate the March cache"
print("setup OK, March cache found at", MARCH_CACHE)

setup OK, March cache found at ../outputs/w05_march_features.parquet


## 1. Two paper findings + my methodology questions

*Pick two findings from the FlyRank research paper. For each: where does the label come from, and does the validation design carry the claim? Constructive tone.*

Picked two findings closest to Lane 3's own territory (segmenting pages into groups and reading
outcomes per group), so the audit doubles as a check on my own method.

### Finding #7 — "The Winning Combinations" (p.13)

**Claim:** health score ordered cleanly across 8 intent × competition-level buckets —
transactional × low leads, informational × medium trails.

**Why this is the relevant comparison for Lane 3.** This is the paper doing archetyping by hand —
segment pages by a combination of categorical features, then read the outcome per segment. That
is structurally the same move as clustering-then-profiling, just with hand-picked bins instead of
learned centroids.

**My methodology question.** Bucket sizes aren't shown next to the health-score bars, and a
transactional-intent, low-competition bucket is very likely smaller than an
informational-medium bucket. Small buckets let a handful of outlier pages move the average a lot,
which can produce a tidy-looking ordering that isn't really an intent × competition effect. This
is exactly why ML-08's cluster search rejects any candidate cluster below 2% of pages before
trusting its centroid — did the eight buckets here clear a similar size floor, and would the
ordering survive it?

### Myth #3 — "Content With Flags Is Failing" (p.20, reversed)

**Claim:** content with 2–3 active optimization flags scores *higher* health than content with
zero flags; flags should be read as workflow leverage, not as evidence of a failing page.

**Where the label comes from.** The paper's own explanation for the pattern is that "many flags
can only trigger on pages that already have enough impressions or behavior data to diagnose a
concrete issue" — i.e. the flag is a decision made by an existing system, and that system can only
fire on pages that were already visible enough to be measured. This is exactly the "decision-derived
feature" pattern our own leakage taxonomy warns about: a flag encodes someone (or something) having
already looked at the page and formed a judgment, so it is a candidate baseline, never a feature.

**My methodology question.** If flagged pages are pre-selected for already having enough
impressions to be diagnosable, then "flags predict higher health" and "pages that already had
traffic score higher" may be the same fact stated twice. Does the flag effect survive controlling
for pre-existing impression volume — the same move our own O/E analysis makes before crediting an
archetype with anything (ML-08 found the ML-07 rule's whole apparent lift was exactly this kind of
confound, hiding inside March CTR)?

In [2]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
import pandas as pd

paper_findings = pd.DataFrame([
    {"finding": "#7 Winning Combinations (p.13)",
     "claim": "health ordered cleanly across 8 intent x competition buckets",
     "bucket_n_disclosed": False,
     "my_question": "does the ordering survive a min-bucket-size floor, "
                     "like ML-08's 2%-of-pages cluster guard?"},
    {"finding": "Myth #3 Flags (p.20, reversed)",
     "claim": "2-3 active flags score higher health than 0 flags",
     "bucket_n_disclosed": False,
     "my_question": "does the effect survive controlling for pre-existing "
                     "impression volume, the way ML-08's O/E analysis "
                     "controls for pre-existing March CTR?"},
])
print(paper_findings.to_string(index=False))
print("\nBoth findings report health scores per bucket with no bucket size printed "
      "alongside them in the paper — the size check itself is what my questions are asking for.")

                       finding                                                        claim  bucket_n_disclosed                                                                                                                               my_question
#7 Winning Combinations (p.13) health ordered cleanly across 8 intent x competition buckets               False                                                does the ordering survive a min-bucket-size floor, like ML-08's 2%-of-pages cluster guard?
Myth #3 Flags (p.20, reversed)            2-3 active flags score higher health than 0 flags               False does the effect survive controlling for pre-existing impression volume, the way ML-08's O/E analysis controls for pre-existing March CTR?

Both findings report health scores per bucket with no bucket size printed alongside them in the paper — the size check itself is what my questions are asking for.


## 2. My model under an honest split (before/after)

*Re-run your Week-5 model under a grouped or time-aware split. Show both numbers.*

Two axes, both new relative to ML-08 (which only ever used `GroupKFold`):

**Axis A — grouped vs random.** ML-08 never actually printed the random-split number to compare
against. Here: out-of-fold cluster assignment under plain row-shuffled `KFold` vs `GroupKFold` by
client, each compared to a reference model fit once on all of March via Adjusted Rand Index. The
gap between the two ARIs is the finding the leakage skill asks for directly — how much of the
"random" number was client memorization.

**Axis B — time-aware, March → May.** The March-fitted scaler and centroids are frozen and used
to score May pages that were never touched during fitting — the same discipline ML-08 already
applies to clients, extended to time. Metric: **retention rate** — the % of pages present and
eligible in both months whose archetype label matches March → May, overall and per archetype
(row-normalized 4×4 confusion matrix), against the **base rate** an independent (chance) relabel
would produce. Also reports the March→May survival rate (the same shape of attrition ML-08 found
in the April window).

In [3]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.model_selection import KFold, GroupKFold
from sklearn.metrics import adjusted_rand_score
from scipy.optimize import linear_sum_assignment

# --- Rebuild March population, identical filters to ML-08 ---
march = pd.read_parquet(MARCH_CACHE)
pop2 = march[march.is_published & (~march.is_deleted)
             & (march.total_impressions > 0) & (march.avg_position > 0)
             & (march.total_impressions >= VOL_FLOOR)].copy().reset_index(drop=True)
pop2["ctr"] = pop2.total_clicks / pop2.total_impressions * 100
X2 = pd.DataFrame({"log_impressions": np.log1p(pop2.total_impressions),
                   "log_clicks": np.log1p(pop2.total_clicks),
                   "ctr": pop2.ctr,
                   "avg_position": pop2.avg_position})[FEATURES]

# Reference archetype model: one fit on ALL of March — the "production" model, frozen from here on
ref_scaler = StandardScaler().fit(X2)
ref_km = KMeans(n_clusters=K_CLUSTERS, random_state=SEED, n_init=10).fit(ref_scaler.transform(X2))
ref_labels = ref_km.labels_
pop2["march_cluster"] = ref_labels
print("reference centroids (original units):")
print(pd.DataFrame(ref_scaler.inverse_transform(ref_km.cluster_centers_), columns=FEATURES).round(2))
print("cluster sizes:", np.bincount(ref_labels), "| n =", len(X2))

# --- Sanity check: match reference clusters to ML-08's already-named archetypes ---
# ML-08's out-of-fold median centroid profile, copied from that notebook's saved output
# (the fitted model object itself isn't persisted to disk, only these printed numbers).
ml08 = pd.DataFrame({
    "archetype":    ["Overlooked", "Steady performers", "Long tail", "Buried"],
    "action":       ["improve/expand", "protect/monitor", "monitor", "prune/rewrite"],
    "n_ml08":       [6546, 30236, 52062, 12565],
    "ctr_med_ml08": [1.172, 0.277, 0.000, 0.000],
    "pos_med_ml08": [6.352, 5.736, 8.225, 41.422],
})
ref_profile = (pop2.groupby("march_cluster")
               .agg(n=("content_hash_id", "size"), ctr_med=("ctr", "median"),
                    pos_med=("avg_position", "median")).reset_index())
print("\nthis notebook's reference clusters:")
print(ref_profile.round(3).to_string(index=False))
print("\nML-08's named archetypes:")
print(ml08.to_string(index=False))

# Match is unambiguous on cluster size alone (all four within <1%), confirmed by matching
# ctr/position ordering: median clicks were used in ML-08, means here, so absolute values
# differ, but the RANK ORDER across clusters (Overlooked highest ctr, Buried deepest
# position, Long tail near-zero ctr) is identical in both tables.
ARCHETYPES = {0: "Steady performers", 1: "Long tail", 2: "Buried", 3: "Overlooked"}
pop2["archetype"] = pop2.march_cluster.map(ARCHETYPES)
print("\nmapping used from here on:", ARCHETYPES)

# --- Axis A: grouped vs random, out-of-fold, ARI against the reference labels ---
def align_to_ref(sc, km):
    cent = ref_scaler.transform(pd.DataFrame(sc.inverse_transform(km.cluster_centers_), columns=FEATURES))
    cost = np.linalg.norm(cent[:, None, :] - ref_km.cluster_centers_[None, :, :], axis=2)
    _, mapping = linear_sum_assignment(cost)
    return mapping

def oof_assign(split_iter):
    labels = np.full(len(X2), -1)
    for tr, te in split_iter:
        sc = StandardScaler().fit(X2.iloc[tr])
        km = KMeans(n_clusters=K_CLUSTERS, random_state=SEED, n_init=10).fit(sc.transform(X2.iloc[tr]))
        mapping = align_to_ref(sc, km)
        labels[te] = mapping[km.predict(sc.transform(X2.iloc[te]))]
    return labels

kf_labels = oof_assign(KFold(n_splits=N_FOLDS, shuffle=True, random_state=SEED).split(X2))
gkf_labels = oof_assign(GroupKFold(n_splits=N_FOLDS).split(X2, groups=pop2.client_hash_id.values))

ari_random, ari_grouped = adjusted_rand_score(ref_labels, kf_labels), adjusted_rand_score(ref_labels, gkf_labels)
print(f"\n=== Axis A: OOF cluster labels vs reference, Adjusted Rand Index ===")
print(f"random KFold  ARI: {ari_random:.4f}")
print(f"GroupKFold    ARI: {ari_grouped:.4f}")
print(f"gap (random - grouped): {ari_random - ari_grouped:+.4f}")

# --- Axis B: time-aware, March -> May, frozen reference scaler/centroids score a month never fit on ---
MAY_CACHE = f"{OUTD}/w06_may_features.parquet"
if os.path.exists(MAY_CACHE):
    may = pd.read_parquet(MAY_CACHE)
else:
    may = _hf_con().sql(f"""
        SELECT content_hash_id,
               SUM(gsc_impressions) AS total_impressions,
               SUM(gsc_clicks)      AS total_clicks,
               SUM(gsc_impressions * gsc_avg_position)
                 / NULLIF(SUM(gsc_impressions), 0) AS avg_position
        FROM read_parquet('{BASE}/fact_content_daily_performance/month=2026-05/*.parquet')
        GROUP BY content_hash_id""").df()
    may.to_parquet(MAY_CACHE, index=False)

may_elig = may[(may.total_impressions > 0) & (may.avg_position > 0)
               & (may.total_impressions >= VOL_FLOOR)].copy()
may_elig["ctr"] = may_elig.total_clicks / may_elig.total_impressions * 100

inter = pop2[["content_hash_id", "march_cluster", "archetype"]].merge(may_elig, on="content_hash_id", how="inner")
X_may = pd.DataFrame({"log_impressions": np.log1p(inter.total_impressions),
                      "log_clicks": np.log1p(inter.total_clicks),
                      "ctr": inter.ctr,
                      "avg_position": inter.avg_position})[FEATURES]
inter["may_cluster"] = ref_km.predict(ref_scaler.transform(X_may))
inter["may_archetype"] = inter.may_cluster.map(ARCHETYPES)

print(f"\n=== Axis B: March -> May migration ===")
print(f"March-eligible: {len(pop2)} | May-eligible: {len(may_elig)} | intersection: {len(inter)}"
      f" | survival: {len(inter)/len(pop2):.1%}")

retention = (inter.march_cluster == inter.may_cluster).mean()
conf = pd.crosstab(inter.archetype, inter.may_archetype, normalize="index") \
         .reindex(index=list(ARCHETYPES.values()), columns=list(ARCHETYPES.values()), fill_value=0)
march_share = inter.archetype.value_counts(normalize=True)
may_share = inter.may_archetype.value_counts(normalize=True).reindex(march_share.index, fill_value=0)
chance = (march_share * may_share).sum()

print(f"overall retention: {retention:.1%}  (n={len(inter)})")
print(f"base rate (expected retention under independence): {chance:.1%}")
print("\nrow-normalized confusion, March archetype (rows) -> May archetype (cols):")
print(conf.round(3).to_string())
print("\nper-archetype retention (diagonal):")
print(pd.Series(np.diag(conf), index=conf.index).round(3).to_string())

reference centroids (original units):
   log_impressions  log_clicks   ctr  avg_position
0             8.45        2.55  0.33          9.44
1             6.16        0.43  0.13         10.31
2             5.88        0.19  0.06         45.78
3             6.27        2.11  1.40          8.50
cluster sizes: [30414 51904 12670  6421] | n = 101409

this notebook's reference clusters:
 march_cluster     n  ctr_med  pos_med
             0 30414    0.278    5.780
             1 51904    0.000    8.197
             2 12670    0.000   41.304
             3  6421    1.178    6.269

ML-08's named archetypes:
        archetype          action  n_ml08  ctr_med_ml08  pos_med_ml08
       Overlooked  improve/expand    6546         1.172         6.352
Steady performers protect/monitor   30236         0.277         5.736
        Long tail         monitor   52062         0.000         8.225
           Buried   prune/rewrite   12565         0.000        41.422

mapping used from here on: {0: 'Steady perf


=== Axis A: OOF cluster labels vs reference, Adjusted Rand Index ===
random KFold  ARI: 0.9922
GroupKFold    ARI: 0.9380
gap (random - grouped): +0.0542



=== Axis B: March -> May migration ===
March-eligible: 101409 | May-eligible: 112309 | intersection: 81547 | survival: 80.4%
overall retention: 65.8%  (n=81547)
base rate (expected retention under independence): 33.0%

row-normalized confusion, March archetype (rows) -> May archetype (cols):
may_archetype      Steady performers  Long tail  Buried  Overlooked
archetype                                                          
Steady performers              0.648      0.245   0.053       0.054
Long tail                      0.067      0.671   0.202       0.060
Buried                         0.026      0.134   0.824       0.016
Overlooked                     0.267      0.318   0.061       0.353

per-archetype retention (diagonal):
archetype
Steady performers    0.648
Long tail            0.671
Buried               0.824
Overlooked           0.353


### Reading the before/after

**Sanity check first.** The reference clusters' median CTR and position match ML-08's OOF-aligned
archetype table almost to the decimal (e.g. Steady performers: 0.278 vs 0.277 CTR, 5.78 vs 5.74
position; sizes within <1%). This is a single full-data fit rather than ML-08's Hungarian-aligned
OOF assignment, and it recovers the same four groups — a mild reassurance about the method's
stability on its own, before either honest-split test below touches it.

**Axis A — grouped vs random: a real but modest gap.** ARI drops from 0.992 (random KFold) to
0.938 (GroupKFold) against the reference labels. Some of what looked like "clustering skill" under
a random split was client memorization, but the drop is small — this archetype set is not mostly
reading off which client a page belongs to, which is a mild point in its favor relative to ML-08's
own worry about single-client clusters.

**Axis B — time-aware, and this is the bigger result.** Retention (65.8%) clears the chance base
rate (33.0%) by a wide margin, so the archetype label is carrying real information about the page
one month forward, not noise. But per-archetype retention is very uneven, and **the direction is
the opposite of what I expected going in**:

| archetype | March→May retention | ML-08's action |
|---|---|---|
| Buried | **82.4%** | prune/rewrite |
| Long tail | 67.1% | monitor |
| Steady performers | 64.8% | protect/monitor |
| Overlooked | **35.3%** | improve/expand |

`Buried` is the *most* durable label of the four — a page classified Buried in March has an 82%
chance of still being Buried in May. That is a second, independent argument for ML-08's
"prune/rewrite" action beyond the O/E-below-chance result: these pages are not drifting away from
the archetype on their own, so waiting is unlikely to resolve the state either.

`Overlooked` is the *least* durable — a majority of pages there in March (65%) have already moved
to a different archetype (mostly Steady or Long tail) by May, before its defining trait, unusually
high CTR on low volume, has had time to compound into anything actionable. Combined with ML-08's
own O/E result for this group (0.896, CI straddling 1.0 — "indistinguishable from chance"), the
honest read is that `Overlooked → improve/expand` is the weakest-supported action of the four: not
wrong, but resting on the least stable archetype and the least conclusive outcome evidence.

## 3. Leakage audit

*The same hunt from Week 3, on your final feature set.*

Five checks, against the attack checklist:

1. **Optimization flags** (`last_optimized_date`) — decision-derived, confirmed absent from
   `FEATURES`. Then the robustness re-run promised in project notes: 612 March pages were
   optimized inside the April outcome window; re-run the O/E comparison excluding them.
2. **`content_updated_date`** — live/mutable, confirmed excluded, with the receipt (distinct
   values, max date past the data window).
3. **`content_hash_id` grain probe** — re-assert content→client is 1:1 (precondition for every
   client-grouped split in this project).
4. **Population selection on outcome-window information** — the published `elig` population
   requires `april_impressions >= VOL_FLOOR`, which is itself a fact about April. Quantify the
   attrition by archetype and show what the improve-rate looks like under a worst-case fill
   instead of silently dropping non-survivors.
5. **Positive control** — inject `april_ctr` as a clustering feature and confirm the harness
   reacts (separation should blow open far past the honest spread). If it doesn't, the test
   rig itself is broken, not the model.

In [4]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

# --- 1. Decision-derived features: confirm absence ---
print("=== 1. Optimization flags / decision-derived features ===")
print("FEATURES used for clustering:", FEATURES)
assert not any("optim" in f.lower() for f in FEATURES)
print("PASS: no optimization-flag or decision-derived column is a clustering feature.\n")

# --- 2. content_updated_date: confirm excluded, show the receipt ---
OPT_CACHE = f"{OUTD}/w06_optimization_flags.parquet"
opt = pd.read_parquet(OPT_CACHE)
print("=== 2. content_updated_date: excluded, mutable/live column ===")
print(f"distinct values: {opt.content_updated_date.nunique()} | "
      f"max date: {opt.content_updated_date.max().date()} (past the March data window)")
print("Not a feature: confirmed live/mutable, no SCD version in this release.\n")

# --- 3. content_hash_id grain probe ---
print("=== 3. content_hash_id -> client_hash_id grain probe ===")
assert march.content_hash_id.duplicated().sum() == 0
assert (march.n_clients > 1).sum() == 0
print("PASS: content_hash_id is unique, maps to exactly one client_hash_id.\n")

# --- 4. Optimization-flag robustness re-run: exclude pages optimized inside April ---
print("=== 4. Optimization-flag robustness re-run ===")
POS_BINS = [-1, 3, 10, 20, 50, 10_000]
POS_LABELS = ["top_3", "page_1", "striking", "page_3_5", "deep"]
pop3 = pop2.copy()
pop3["log_impressions"] = np.log1p(pop3.total_impressions)
pop3["log_clicks"] = np.log1p(pop3.total_clicks)
pop3["pos_tier"] = pd.cut(pop3.avg_position, bins=POS_BINS, labels=POS_LABELS)
folds = list(GroupKFold(n_splits=N_FOLDS).split(X2, groups=pop3.client_hash_id.values))
pop3["peer_median_ctr"] = np.nan
for tr, te in folds:
    med = pop3.iloc[tr].groupby("pos_tier", observed=True)["ctr"].median()
    pop3.loc[pop3.index[te], "peer_median_ctr"] = pop3.iloc[te].pos_tier.map(med).astype(float).values
gap = pop3.peer_median_ctr - pop3.ctr
pop3["baseline_score"] = np.where(gap > 0, pop3.total_impressions * gap / 100, 0.0)
pop3["baseline_action"] = np.where(gap > 0, "snippet_fix", "monitor")

april = pd.read_parquet(f"{OUTD}/w05_april_outcome.parquet")
pop3 = pop3.merge(april, on="content_hash_id", how="left")
pop3["april_ctr"] = np.where(pop3.april_impressions > 0,
                              pop3.april_clicks / pop3.april_impressions * 100, np.nan)

elig = pop3[pop3.april_impressions >= VOL_FLOOR].copy()
elig["improved"] = elig.april_ctr > elig.ctr

opt_window = opt[(opt.last_optimized_date >= "2026-04-01") & (opt.last_optimized_date <= "2026-04-30")]
n_raw = march.content_hash_id.isin(opt_window.content_hash_id).sum()
n_march_elig = pop3.content_hash_id.isin(opt_window.content_hash_id).sum()
n_flagged = elig.content_hash_id.isin(opt_window.content_hash_id).sum()
print(f"pages optimized inside the April outcome window: {n_raw} across all of March's raw data "
      f"(matches the 612 in project notes) -> {n_march_elig} within the March-modeling population "
      f"-> {n_flagged} within the published April-outcome-eligible population (elig)")
excl_opt = elig[~elig.content_hash_id.isin(opt_window.content_hash_id)]

def improve_table(df, label):
    base = df.improved.mean()
    rows = [{"group": "ALL eligible", "n": len(df), "improve_rate": base}]
    r = df[df.baseline_action == "snippet_fix"]
    rows.append({"group": "RULE: snippet_fix", "n": len(r), "improve_rate": r.improved.mean()})
    for a in ARCHETYPES.values():
        s = df[df.archetype == a]
        rows.append({"group": f"MODEL: {a}", "n": len(s), "improve_rate": s.improved.mean()})
    out = pd.DataFrame(rows)
    out["lift_vs_base"] = out.improve_rate / base
    out.insert(0, "run", label)
    return out

before = improve_table(elig, "before (all elig)")
after = improve_table(excl_opt, "after (excl. optimized-in-window)")
print(pd.concat([before, after], ignore_index=True).round(4).to_string(index=False))

# --- 5. Population selection on outcome-window information ---
print("\n=== 5. Population selection depends on the April (outcome) window ===")
pop3["survived_april_floor"] = pop3.april_impressions.fillna(0) >= VOL_FLOOR
surv = pop3.groupby("archetype")["survived_april_floor"].mean()
print(f"March-eligible: {len(pop3)} | published elig (requires April floor): {len(elig)}"
      f" ({len(elig)/len(pop3):.1%})")
print("\nsurvival rate to the April volume floor, by archetype:")
print(surv.round(4).to_string())

pop3["improved_worstcase"] = np.where(pop3.survived_april_floor, pop3.april_ctr > pop3.ctr, False)
rows = [{"group": "ALL eligible", "improve_rate": pop3.improved_worstcase.mean()}]
for a in ARCHETYPES.values():
    s = pop3[pop3.archetype == a]
    rows.append({"group": f"MODEL: {a}", "improve_rate": s.improved_worstcase.mean()})
worst = pd.DataFrame(rows)
comp = worst.merge(before[["group", "improve_rate"]].rename(columns={"improve_rate": "published_rate"}),
                    on="group", how="left")
comp["gap_worstcase_minus_published"] = comp.improve_rate - comp.published_rate
print("\nworst-case (non-survivors counted as NOT improved) vs published (survivors only):")
print(comp.round(4).to_string(index=False))

# --- 6. Positive control: inject april_ctr as a feature and confirm the harness reacts ---
# Clustering is unsupervised and doesn't optimize for any outcome, so leaking a feature into
# KMeans is not guaranteed to separate an outcome label cleanly - that would be the wrong test.
# The honest test: can a simple SUPERVISED model predict "improved" better once it can see the
# label's own ingredient? Train once WITHOUT april_ctr, once WITH it, same rows, same CV.
print("\n=== 6. Positive control: predict 'improved' with vs without the leaked ingredient ===")
from sklearn.tree import DecisionTreeClassifier
from sklearn.model_selection import cross_val_score, StratifiedKFold

leak_pop = elig.dropna(subset=["april_ctr"]).copy()
y = leak_pop.improved.astype(int)
X_honest = leak_pop[FEATURES]
X_leaked = leak_pop[FEATURES + ["april_ctr"]]
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)
auc_honest = cross_val_score(DecisionTreeClassifier(max_depth=4, random_state=SEED),
                             X_honest, y, cv=cv, scoring="roc_auc").mean()
auc_leaked = cross_val_score(DecisionTreeClassifier(max_depth=4, random_state=SEED),
                             X_leaked, y, cv=cv, scoring="roc_auc").mean()
print(f"base rate (improved=1 share): {y.mean():.4f}")
print(f"cross-validated AUC, honest features only:      {auc_honest:.4f}")
print(f"cross-validated AUC, honest + LEAKED april_ctr:  {auc_leaked:.4f}")
verdict = "PASS: harness is sensitive - leaked AUC collapses toward 1.0" if auc_leaked > auc_honest + 0.15 \
          else "FAIL: leak did not move the score - harness may be broken"
print(verdict)
print("The leaked feature is discarded here; only the honest, March-only archetypes and rule "
      "are used anywhere else in this project.")

=== 1. Optimization flags / decision-derived features ===
FEATURES used for clustering: ['log_impressions', 'log_clicks', 'ctr', 'avg_position']
PASS: no optimization-flag or decision-derived column is a clustering feature.



=== 2. content_updated_date: excluded, mutable/live column ===
distinct values: 242 | max date: 2026-07-06 (past the March data window)
Not a feature: confirmed live/mutable, no SCD version in this release.

=== 3. content_hash_id -> client_hash_id grain probe ===
PASS: content_hash_id is unique, maps to exactly one client_hash_id.

=== 4. Optimization-flag robustness re-run ===


pages optimized inside the April outcome window: 612 across all of March's raw data (matches the 612 in project notes) -> 423 within the March-modeling population -> 235 within the published April-outcome-eligible population (elig)


                              run                    group     n  improve_rate  lift_vs_base
                before (all elig)             ALL eligible 88474        0.2925        1.0000
                before (all elig)        RULE: snippet_fix 36704        0.3335        1.1404
                before (all elig) MODEL: Steady performers 30365        0.3273        1.1191
                before (all elig)         MODEL: Long tail 42740        0.3100        1.0599
                before (all elig)            MODEL: Buried  9789        0.1904        0.6511
                before (all elig)        MODEL: Overlooked  5580        0.1477        0.5049
after (excl. optimized-in-window)             ALL eligible 88239        0.2929        1.0000
after (excl. optimized-in-window)        RULE: snippet_fix 36561        0.3344        1.1418
after (excl. optimized-in-window) MODEL: Steady performers 30336        0.3273        1.1176
after (excl. optimized-in-window)         MODEL: Long tail 42561      


worst-case (non-survivors counted as NOT improved) vs published (survivors only):
                   group  improve_rate  published_rate  gap_worstcase_minus_published
            ALL eligible        0.2552          0.2925                        -0.0373
MODEL: Steady performers        0.3268          0.3273                        -0.0005
        MODEL: Long tail        0.2553          0.3100                        -0.0547
           MODEL: Buried        0.1471          0.1904                        -0.0433
       MODEL: Overlooked        0.1283          0.1477                        -0.0193

=== 6. Positive control: predict 'improved' with vs without the leaked ingredient ===


base rate (improved=1 share): 0.2925
cross-validated AUC, honest features only:      0.6366
cross-validated AUC, honest + LEAKED april_ctr:  0.9396
PASS: harness is sensitive - leaked AUC collapses toward 1.0
The leaked feature is discarded here; only the honest, March-only archetypes and rule are used anywhere else in this project.


### Reading the audit

**Checks 1–3 pass as expected.** No decision-derived column is a feature, `content_updated_date`
stays excluded with the same receipt already on file (242 distinct values, max date past the
window), and the content→client grain holds — a precondition every grouped split in this project
depends on.

**Check 4 — the 612-page robustness re-run changes nothing.** The count reconciles cleanly: 612
pages across all of March's raw data (matching project notes) narrow to 423 in the modeling
population, then 235 that also clear the April outcome floor. Excluding those 235 moves every
archetype's improve-rate by at most 0.001 and every lift figure in the third decimal (Buried
0.6511→0.6506, Overlooked 0.5049→0.5047). ML-08's conclusions do not depend on these pages being
in the sample.

**Check 5 is the one worth carrying forward.** The published `elig` population silently drops
pages that fall below the April volume floor — an outcome-window fact deciding who counts. Survival
rate to that floor is uneven by archetype: Buried 77.3% (matches ML-08's own Limitations note of
76.9% almost exactly), Long tail 82.3%, Overlooked 86.9%, Steady performers 99.8%. Filling
non-survivors in as "did not improve" (the worst-case, conservative accounting) instead of
dropping them shaves the improve-rate down most for Long tail (−5.5pp) and Buried (−4.3pp), and
barely touches Steady (−0.05pp) — consistent with its near-total survival. **Nothing reorders**:
Buried stays the worst-performing archetype and Steady stays the best under either accounting, so
this doesn't overturn ML-08's ranking. But it does mean the published numbers for Long tail and
Buried specifically are mildly optimistic, and that optimism was previously undisclosed rather
than quantified.

**Check 6 — the harness is honest.** Predicting `improved` from the four honest features alone
gets AUC 0.637 (real signal — March CTR position in the distribution says something about April
movement even honestly). Adding the leaked `april_ctr` collapses it to 0.940. The test rig
reacts the way a working leakage test should; a version of this check that used unsupervised
KMeans (leaking the feature into clustering rather than into a model that predicts the outcome)
was tried first and produced no clear separation — worth recording as a lesson: the leak-and-watch-
it-jump test only works when there's an actual target being predicted, not when the model doesn't
optimize for any label at all.

## 4. Claim rewrite

*Take your own boldest sentence and rewrite it in safe language: observed, measured, directional, decision-support.*

**Original (ML-08, Section 4):** *"Buried pages recover markedly less than their CTR-matched
peers (O/E 0.645). That is real evidence that buried pages do not self-heal — which is a genuine
argument for prune/rewrite over monitor, and the most decision-relevant result in the notebook."*

Pulling together everything this notebook has independently measured about the same claim before
rewriting it.

In [5]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

# Recompute the Buried O/E + CI directly in THIS notebook (cross-check against ML-08's 0.645,
# not just a citation) using the elig/pop3 objects already built and audited in Section 3.
print("=== Claim rewrite: supporting numbers for the Buried claim ===")
rng = np.random.default_rng(SEED)
elig["stratum"] = np.where(elig.ctr == 0, "zero", "")
nz = elig.ctr > 0
elig.loc[nz, "stratum"] = pd.qcut(elig.loc[nz, "ctr"], 8, duplicates="drop").astype(str)
strata = elig.groupby("stratum").agg(base_improved=("improved", "mean"))
elig["exp"] = elig.stratum.map(strata.base_improved)
clients = elig.client_hash_id.unique()
by_client = {c: gg for c, gg in elig.groupby("client_hash_id")}

def oe(idx, B=300):
    gsub = elig.loc[idx]
    point = gsub.improved.mean() / gsub["exp"].mean()
    boots = []
    for _ in range(B):
        s = pd.concat([by_client[c] for c in rng.choice(clients, len(clients), replace=True)])
        s = s[s.index.isin(gsub.index)]
        if len(s) > 30 and s["exp"].mean() > 0:
            boots.append(s.improved.mean() / s["exp"].mean())
    lo, hi = np.percentile(boots, [2.5, 97.5])
    return point, lo, hi

buried_idx = elig.index[elig.archetype == "Buried"]
oe_point, oe_lo, oe_hi = oe(buried_idx)

print(f"Buried O/E, recomputed here on the published elig population: "
      f"{oe_point:.3f}  95% CI [{oe_lo:.3f}, {oe_hi:.3f}]  (ML-08 reported 0.645, [0.456, 0.810])")
print(f"Buried March -> May archetype retention (Section 2):             0.824")
print(f"Buried survival to the April volume floor (Section 3, check 5):  0.773")
print(f"Buried improve-rate, published (survivors only):                 "
      f"{elig.loc[buried_idx, 'improved'].mean():.4f}")
print(f"Buried improve-rate, worst-case (non-survivors = not improved):  "
      f"{pop3.loc[pop3.archetype == 'Buried', 'improved_worstcase'].mean():.4f}")
print(f"Robustness re-run (check 4, excl. optimized-in-window): O/E lift barely moves, "
      f"0.6511 -> 0.6506")

=== Claim rewrite: supporting numbers for the Buried claim ===


Buried O/E, recomputed here on the published elig population: 0.667  95% CI [0.470, 0.855]  (ML-08 reported 0.645, [0.456, 0.810])
Buried March -> May archetype retention (Section 2):             0.824
Buried survival to the April volume floor (Section 3, check 5):  0.773
Buried improve-rate, published (survivors only):                 0.1904
Buried improve-rate, worst-case (non-survivors = not improved):  0.1471
Robustness re-run (check 4, excl. optimized-in-window): O/E lift barely moves, 0.6511 -> 0.6506


### The rewrite

**Original:** *"Buried pages recover markedly less than their CTR-matched peers (O/E 0.645). That
is real evidence that buried pages do not self-heal — which is a genuine argument for
prune/rewrite over monitor, and the most decision-relevant result in the notebook."*

**Rewritten:** In the observed March→April window, pages classified `Buried` recovered less than
CTR-matched peers (O/E 0.667, 95% CI [0.470, 0.855] here; ML-08 independently reported 0.645,
[0.456, 0.810] — both intervals sit below 1.0). This is a measured association in one
non-intervention window, not evidence of causation, and it likely understates the true gap: only
77.3% of Buried pages survived to the April measurement floor, and the missing 23% are plausibly
the ones that lost the most traffic, not a random subset — so the O/E figure is computed on the
more-visible half of an already weak-performing group. The `Buried` label is also the most durable
of the four archetypes measured (82.4% of pages keep the label from March to May), so this is not
a state pages are likely to exit on their own before any review could happen. Excluding the 235
`Buried`-eligible pages that received a workflow optimization inside the April window barely moves
either number, so neither result depends on that overlap. Taken together, this is directional,
decision-support evidence — two independently measured signals pointing the same way, in the same
non-random population, over one window — for prioritizing `Buried` pages for prune/rewrite review
ahead of a wait-and-monitor approach. It is not proof that rewriting these specific pages will
work, and no causal claim is intended or available from this data.

## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.

**Reproducibility.** Seed fixed at 42 throughout (KMeans, KFold/GroupKFold, bootstrap RNG).
Warehouse scans are cached to `work/outputs/*.parquet` (gitignored) and re-pulled automatically if
absent; this notebook reuses w05's March/April caches and adds two of its own (May features,
`dim_content` optimization/update dates). Reproduces from a clean checkout once w05 has been run
once to populate its caches.